
# Reconocimiento de Entidades Nombradas (NER)

## 1. Introducción

Este notebook implementa el reconocimiento de entidades nombradas (Named Entity Recognition, NER) sobre noticias recopiladas mediante fuentes RSS.

El objetivo es identificar y clasificar entidades relevantes para el monitoreo de medios de comunicación, permitiendo reconocer personas, organizaciones, lugares y productos o servicios mencionados en las noticias.

Para esta tarea se utiliza **GLiNER Multilingual v2.1**, un modelo multilingüe que permite definir las categorías de entidades durante la inferencia, sin necesidad de realizar un entrenamiento adicional.

El procesamiento se realiza localmente, utilizando el título y la descripción de cada noticia almacenada en PostgreSQL.

Los resultados se validarán y posteriormente se almacenarán en la base de datos para su utilización en los componentes de análisis y visualización del asistente.


In [1]:
import pandas as pd
import numpy as np
import psycopg
import os
import logging
from transformers import pipeline
from gliner import GLiNER
from dotenv import load_dotenv

e:\Trabajo final UGR\communication-intelligence-assistant\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


### Configuración de la conexión experimental

Se establece la conexión con la base de datos del caso
experimental, garantizando el aislamiento respecto del
entorno original del asistente.

In [2]:
# Configuración de la base de datos experimental.
EXPECTED_DATABASE = "ida_caso_02"
ENV_PATH = "../.env.ida_caso_02"

# Cargar las variables del entorno experimental.
if not load_dotenv(ENV_PATH, override=True):
    raise RuntimeError("No se pudo cargar el entorno experimental.")

In [3]:
def get_database_connection():
    """Crea una conexión con la base de datos PostgreSQL configurada.

    Returns:
        psycopg.Connection: Conexión activa a la base de datos PostgreSQL.
    """
    return psycopg.connect(
        host=os.getenv("DB_HOST"),
        port=os.getenv("DB_PORT"),
        dbname=os.getenv("DB_NAME"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
    )

In [4]:
# Verificar la conexión con la base experimental.
with get_database_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute("SELECT current_database();")
        database = cursor.fetchone()[0]

assert database == EXPECTED_DATABASE, (
    f"Base de datos incorrecta: {database}"
)

print(f"Base experimental activa: {database}")

Base experimental activa: ida_caso_02


In [5]:
# Configuración del registro de eventos.
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)

logger = logging.getLogger(__name__)


# Configuración del modelo NER.
NER_CONFIG = {
    "model_name": "urchade/gliner_multi-v2.1",
    "entity_labels": [
        "person",
        "organization",
        "location",
        "product",
    ],
    "threshold": 0.5,
}



## 2. Carga del modelo NER

Se carga el modelo GLiNER Multilingual v2.1 utilizando la configuración definida previamente.

El modelo permite identificar cuatro categorías de entidades: personas, organizaciones, lugares y productos o servicios.

Antes de procesar las noticias almacenadas en la base de datos, se realiza una prueba controlada para verificar su funcionamiento.


In [6]:

def load_ner_model(config):
    """
    Carga el modelo GLiNER para reconocimiento de entidades.

    Parameters
    ----------
    config : dict
        Configuración del modelo NER.

    Returns
    -------
    GLiNER
        Modelo cargado y preparado para inferencia.
    """
    logger.info("Cargando modelo NER: %s", config["model_name"])

    model = GLiNER.from_pretrained(config["model_name"])
    model.eval()

    logger.info("Modelo NER cargado correctamente.")

    return model


ner_model = load_ner_model(NER_CONFIG)


2026-11-26 20:42:59,915 - INFO - Cargando modelo NER: urchade/gliner_multi-v2.1
e:\Trabajo final UGR\communication-intelligence-assistant\.venv\Lib\site-packages\huggingface_hub\utils\_validators.py:190: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
2026-11-26 20:43:00,324 - INFO - HTTP Request: GET https://huggingface.co/api/models/urchade/gliner_multi-v2.1/revision/main "HTTP/1.1 200 OK"


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

2026-11-26 20:43:00,387 - INFO - Loading the following GLiNER type: <class 'gliner.model.UniEncoderSpanGLiNER'>...
2026-11-26 20:43:00,551 - INFO - HTTP Request: HEAD https://huggingface.co/microsoft/mdeberta-v3-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-11-26 20:43:00,552 - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-11-26 20:43:00,565 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/mdeberta-v3-base/a0484667b22365f84929a935b5e50a51f71f159d/config.json?%2Fmicrosoft%2Fmdeberta-v3-base%2Fresolve%2Fmain%2Fconfig.json=&etag=%2227b905d788d20826414d27ce9fa9765623e38ea7%22 "HTTP/1.1 200 OK"
2026-11-26 20:43:00,740 - INFO - HTTP Request: HEAD https://huggingface.co/microsoft/mdeberta-v3-base/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-11-26 20:43:00,754 - INFO - HTTP Request: HEAD https://huggi


## 3. Recuperación y preparación de noticias

Se recuperan las noticias almacenadas en la tabla `rss_news` de PostgreSQL.

Para el reconocimiento de entidades nombradas se construye un texto a partir del título y la descripción de cada noticia. Esta combinación proporciona mayor contexto al modelo y permite identificar entidades mencionadas en ambos campos.

Se excluyen del procesamiento los registros que no contienen texto válido.


In [7]:

def get_news_for_ner(connection):
    """
    Recupera las noticias disponibles para el procesamiento NER.

    Parameters
    ----------
    connection :
        Conexión activa a PostgreSQL.

    Returns
    -------
    pandas.DataFrame
        Noticias con identificador, título y descripción.
    """
    query = """
        SELECT
            rss_news_id,
            title,
            description
        FROM rss_news
        ORDER BY rss_news_id;
    """

    return pd.read_sql_query(query, connection)


connection = get_database_connection()

try:
    news_df = get_news_for_ner(connection)
finally:
    connection.close()

logger.info("Noticias recuperadas: %s", len(news_df))

display(news_df.head())


C:\Users\enzor\AppData\Local\Temp\ipykernel_32884\386490827.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, connection)
2026-11-26 20:43:09,511 - INFO - Noticias recuperadas: 50


,rss_news_id,title,description
0,1,Energía Costa Serena: amplía la generación solar,La puesta en marcha de 24 MW adicionales fue c...
1,2,Energía Costa Serena: reduce interrupciones el...,La mejora del servicio benefició a 800 hogares...
2,3,Energía Costa Serena: recupera capacidad de un...,La reparación permitió recuperar 18 MW y refor...
3,4,Energía Costa Serena: sufre una falla en una s...,El corte afectó a 1200 usuarios y generó recla...
4,5,Energía Costa Serena: demora obras de transpor...,El retraso de tres meses agravó la incertidumb...


In [8]:

def build_ner_text(row):
    """
    Combina el título y la descripción de una noticia.
    """
    title = str(row["title"]).strip() if pd.notna(row["title"]) else ""
    description = (
        str(row["description"]).strip()
        if pd.notna(row["description"])
        else ""
    )

    return ". ".join(
        part for part in [title, description] if part
    )


news_df["ner_text"] = news_df.apply(
    build_ner_text,
    axis=1,
)

# Excluir noticias sin texto válido.
news_df = news_df.loc[
    news_df["ner_text"].str.strip().ne("")
].copy()

logger.info(
    "Noticias con texto válido: %s",
    len(news_df),
)

display(
    news_df[["rss_news_id", "ner_text"]].head()
)


2026-11-26 20:43:13,937 - INFO - Noticias con texto válido: 50


,rss_news_id,ner_text
0,1,Energía Costa Serena: amplía la generación sol...
1,2,Energía Costa Serena: reduce interrupciones el...
2,3,Energía Costa Serena: recupera capacidad de un...
3,4,Energía Costa Serena: sufre una falla en una s...
4,5,Energía Costa Serena: demora obras de transpor...



## 4. Extracción de entidades nombradas

Se aplica el modelo GLiNER Multilingual v2.1 sobre los textos preparados de las noticias.

Para cada entidad identificada se registra:

- Identificador de la noticia de origen.
- Texto de la entidad.
- Categoría asignada: persona, organización, lugar o producto.
- Puntuación de confianza del modelo.
- Posición inicial y final dentro del texto.

Se utiliza un umbral de confianza de 0,5 para filtrar las predicciones del modelo.

Los resultados se almacenan temporalmente en un DataFrame para su posterior validación y persistencia.


In [9]:

def extract_news_entities(news, model, config):
    """
    Extrae entidades nombradas de las noticias mediante GLiNER.

    Parameters
    ----------
    news : pandas.DataFrame
        Noticias con identificador y texto preparado.
    model : GLiNER
        Modelo NER cargado.
    config : dict
        Configuración del procesamiento.

    Returns
    -------
    pandas.DataFrame
        Entidades identificadas en las noticias.
    """
    records = []

    for row in news.itertuples(index=False):
        entities = model.predict_entities(
            text=row.ner_text,
            labels=config["entity_labels"],
            threshold=config["threshold"],
        )

        for entity in entities:
            entity_text = entity["text"].strip()

            if not entity_text:
                continue

            records.append(
                {
                    "rss_news_id": int(row.rss_news_id),
                    "entity_text": entity_text,
                    "entity_type": entity["label"],
                    "confidence": float(entity["score"]),
                    "start_char": int(entity["start"]),
                    "end_char": int(entity["end"]),
                }
            )

    return pd.DataFrame(
        records,
        columns=[
            "rss_news_id",
            "entity_text",
            "entity_type",
            "confidence",
            "start_char",
            "end_char",
        ],
    )


In [10]:

logger.info(
    "Iniciando extracción NER de %s noticias",
    len(news_df),
)

ner_results = extract_news_entities(
    news=news_df,
    model=ner_model,
    config=NER_CONFIG,
)

logger.info(
    "Procesamiento finalizado. Entidades detectadas: %s",
    len(ner_results),
)

print(f"Noticias procesadas: {len(news_df)}")
print(f"Entidades detectadas: {len(ner_results)}")
print(
    "Noticias con entidades:",
    ner_results["rss_news_id"].nunique(),
)

display(ner_results.head(15))


2026-11-26 20:43:24,144 - INFO - Iniciando extracción NER de 50 noticias
2026-11-26 20:43:29,482 - INFO - Procesamiento finalizado. Entidades detectadas: 55


Noticias procesadas: 50
Entidades detectadas: 55
Noticias con entidades: 40


,rss_news_id,entity_text,entity_type,confidence,start_char,end_char
0,1,Energía Costa Serena,organization,0.932124,0,20
1,2,Energía Costa Serena,organization,0.917243,0,20
2,3,Energía Costa Serena,organization,0.802159,0,20
3,4,Energía Costa Serena,organization,0.824049,0,20
4,4,1200 usuarios,person,0.709975,76,89
5,5,Energía Costa Serena,organization,0.882452,0,20
6,6,Energía Costa Serena,organization,0.919172,0,20
7,6,trabajadores,person,0.574793,116,128
8,7,Energía Costa Serena,organization,0.936901,0,20
9,8,Energía Costa Serena,organization,0.912050,0,20



## 5. Validación de los resultados

Se realizan controles de calidad sobre las entidades extraídas para verificar la consistencia de los resultados antes de su almacenamiento.

Las validaciones incluyen:

- Ausencia de valores nulos y entidades vacías.
- Categorías pertenecientes al conjunto definido.
- Puntuaciones de confianza dentro del rango esperado.
- Correspondencia entre las posiciones de las entidades y el texto original.
- Asociación de cada entidad con una noticia existente.

Estas verificaciones permiten detectar errores estructurales del procesamiento, pero no determinan la precisión semántica del modelo, que será evaluada posteriormente.


In [11]:

def validate_ner_results(results, news, config):
    """
    Valida la integridad y consistencia de las entidades NER.
    """
    required_columns = [
        "rss_news_id",
        "entity_text",
        "entity_type",
        "confidence",
        "start_char",
        "end_char",
    ]

    # Verificar estructura y valores nulos.
    assert set(required_columns).issubset(results.columns)
    assert not results[required_columns].isna().any().any()

    # Verificar entidades y categorías.
    assert results["entity_text"].str.strip().ne("").all()
    assert results["entity_type"].isin(config["entity_labels"]).all()

    # Verificar puntuaciones.
    assert results["confidence"].between(
        config["threshold"], 1.0
    ).all()

    # Verificar identificadores de noticias.
    assert results["rss_news_id"].isin(news["rss_news_id"]).all()

    # Verificar posiciones y correspondencia textual.
    news_texts = news.set_index("rss_news_id")["ner_text"]

    invalid_positions = []

    for row in results.itertuples(index=False):
        text = news_texts.loc[row.rss_news_id]

        valid_range = (
            0 <= row.start_char < row.end_char <= len(text)
        )

        valid_text = (
            text[row.start_char:row.end_char].strip()
            == row.entity_text
        ) if valid_range else False

        if not (valid_range and valid_text):
            invalid_positions.append(row.rss_news_id)

    assert not invalid_positions, (
        f"Entidades con posiciones inconsistentes: "
        f"{invalid_positions[:10]}"
    )

    logger.info("Validaciones NER completadas correctamente.")

    return True


In [12]:

validate_ner_results(
    results=ner_results,
    news=news_df,
    config=NER_CONFIG,
)

print("Distribución de entidades por categoría:")

display(
    ner_results["entity_type"]
    .value_counts()
    .rename_axis("entity_type")
    .reset_index(name="total")
)


2026-11-26 20:43:43,924 - INFO - Validaciones NER completadas correctamente.


Distribución de entidades por categoría:


,entity_type,total
0,organization,39
1,product,6
2,person,5
3,location,5




## 6. Persistencia de resultados

Las entidades reconocidas se almacenan en la tabla `rss_entities` de PostgreSQL.

Se utiliza una estrategia de inserción con control de duplicados mediante `ON CONFLICT`, permitiendo ejecutar nuevamente el procesamiento sin generar registros repetidos.

Cada registro conserva la noticia de origen, el texto y categoría de la entidad, la puntuación de confianza y el modelo utilizado.

La fecha de procesamiento se registra automáticamente en la base de datos.



In [13]:

def save_ner_results(connection, results, model_name):
    """
    Persiste las entidades NER en PostgreSQL evitando duplicados.

    Parameters
    ----------
    connection :
        Conexión activa a PostgreSQL.
    results : pandas.DataFrame
        Entidades extraídas mediante GLiNER.
    model_name : str
        Identificador del modelo utilizado.

    Returns
    -------
    int
        Cantidad de registros insertados.
    """
    query = """
        INSERT INTO rss_entities (
            fk_rss_news_id,
            entity_text,
            entity_type,
            confidence_score,
            model_name
        )
        VALUES (%s, %s, %s, %s, %s)
        ON CONFLICT ON CONSTRAINT uq_rss_entities
        DO NOTHING
        RETURNING rss_entity_id;
    """

    inserted = 0

    try:
        with connection.cursor() as cursor:
            for row in results.itertuples(index=False):
                cursor.execute(
                    query,
                    (
                        int(row.rss_news_id),
                        row.entity_text,
                        row.entity_type,
                        float(row.confidence),
                        model_name,
                    ),
                )

                if cursor.fetchone() is not None:
                    inserted += 1

        connection.commit()

    except Exception:
        connection.rollback()
        logger.exception("Error al persistir las entidades NER.")
        raise

    logger.info("Entidades insertadas: %s", inserted)

    return inserted


In [14]:

connection = get_database_connection()

try:
    inserted_count = save_ner_results(
        connection=connection,
        results=ner_results,
        model_name=NER_CONFIG["model_name"],
    )
finally:
    connection.close()

print(f"Entidades detectadas: {len(ner_results)}")
print(f"Entidades insertadas: {inserted_count}")


2026-11-26 20:44:06,974 - INFO - Entidades insertadas: 55


Entidades detectadas: 55
Entidades insertadas: 55


In [16]:

def get_ner_summary(connection):
    """
    Recupera un resumen de las entidades almacenadas.
    """
    query = """
        SELECT
            entity_type,
            COUNT(*) AS total,
            COUNT(DISTINCT fk_rss_news_id) AS news_count
        FROM rss_entities
        WHERE model_name = %s
        GROUP BY entity_type
        ORDER BY total DESC;
    """

    return pd.read_sql_query(
        query,
        connection,
        params=(NER_CONFIG["model_name"],),
    )


connection = get_database_connection()

try:
    ner_summary = get_ner_summary(connection)
finally:
    connection.close()

display(ner_summary)
print(f"Total almacenado: {ner_summary['total'].sum()}")


C:\Users\enzor\AppData\Local\Temp\ipykernel_32884\1835032258.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(


,entity_type,total,news_count
0,organization,39,38
1,product,6,6
2,location,5,5
3,person,5,5


Total almacenado: 55
